# Jitter engine pipeline demonstration
This notebook demonstrate how to use the full jitter pipeline.  The main engine is contained in the `JitterEvaluator` class of the `jitter` module.  The `JitterEvaluator` class is fundamentally a wrapper of three models:
- a contextual embedder from HuggingFace,
- a logistic regression for relevance filtering,
- a logistic regression for jitter scoring.

In [1]:
import pandas as pd
import ast
from jitter.engine import JitterEvaluator
from jitter.routines import get_headlines

The engine can be instantiated by passing a string describing the HF model.  If no string is passed, it defaults to `ProsusAI/finbert`.

In [2]:
engine = JitterEvaluator()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The engine object needs to be trained on a training dataset.  The training dataset must contain (at least) the following four columns:
- `embedding`: the embedded sentence,
- `relevant`: 0 if not relevant, 1 if relevant,
- `jittery`: 0 if not jittery, 1 if jittery.

At startup, we load a pre-labeled dataset from `training.csv`.  Because CSV cannot hold list types, we need to evaluate the `embedding` string column to a list.

In [ ]:
training = pd.read_csv("data/training.csv")
training["embedding"] = training.embedding.apply(ast.literal_eval)

We can then train the engine with the training dataset.

In [4]:
engine.train(training)

/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


Next, we need some headlines to score.  We use the `get_headlines` function from the `routines` module.  The function takes a list of RSS urls and an optional date parameter, and returns the headlines from the given urls on the given date as `pd.Series`.  The `trim` parameter dictates how many headlines to keep (keeps most recent).

In [ ]:
urls = pd.read_csv("data/sources.csv").url
headlines = await get_headlines(urls, trim=300)

Skipping {'title': 'Asia’s ‘Ukraine Moment’ Is Supercharging the Energy Transition', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://feeds.bloomberg.com/green/news.rss', 'value': 'Asia’s ‘Ukraine Moment’ Is Supercharging the Energy Transition'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://www.bloomberg.com/news/audio/2026-06-04/zero-podcast-asia-s-ukraine-moment-supercharges-clean-energy'}], 'link': 'https://www.bloomberg.com/news/audio/2026-06-04/zero-podcast-asia-s-ukraine-moment-supercharges-clean-energy', 'id': 'https://www.bloomberg.com/news/audio/2026-06-04/zero-podcast-asia-s-ukraine-moment-supercharges-clean-energy', 'guidislink': False, 'authors': [{'name': 'Akshat Rathi'}], 'author': 'Akshat Rathi', 'author_detail': {'name': 'Akshat Rathi'}, 'published': 'Thu, 04 Jun 2026 01:00:00 GMT', 'published_parsed': time.struct_time(tm_year=2026, tm_mon=6, tm_mday=4, tm_hour=1, tm_min=0, tm_sec=0, tm_wday=3, tm_yday=155, tm_isdst=0)}


Once we have the headlines, we process them.  This means that the engine populates an internal dataframe with the headlines, computes their embeddings, estimates which ones are relevant, and predicts which of the relevant headlines are jittery or not.

In [ ]:
engine.process_headlines(headlines)

The `current_prediction` property of engine gives the scores for all headlines.

In [ ]:
engine.current_prediction

,concat,embedding,relevant,jitter
0,Iran War Live Updates: Israel Strikes Southern...,"[-0.0881357192993164, 0.22378650307655334, -0....",1,0.762777
1,Russia Launches Deadly Strikes on Kyiv After T...,"[-0.000947926368098706, 0.20662473142147064, 0...",1,0.914559
2,"U.S. Ebola Unit Plans in Kenya, Subject of Pro...","[-0.13119040429592133, -0.2009671926498413, -0...",1,0.83135
3,The New Zealand Parakeet Pair That Are Saving ...,"[0.5227239727973938, 0.19449959695339203, -0.0...",0,NaN
4,U.S. Was Asked to Blacklist Colombian Cartel G...,"[0.23318034410476685, 0.1427941769361496, -0.2...",1,0.746284
...,...,...,...,...
262,"Lebanon Tensions, Commerzbank's 30% UniCredit ...","[0.19070670008659363, -0.3762185275554657, 0.0...",1,0.573668
263,Incoming Tepco Chair Open to Alliance Partners...,"[-0.04652702808380127, -0.005580492317676544, ...",1,0.530587
264,US Nuclear Fuel Enricher Scales Up to Offset R...,"[0.37307125329971313, 0.27025938034057617, -0....",1,0.475826
265,"Data Centers Now Use So Much Power, Their Desi...","[0.09473615884780884, 0.7131877541542053, -0.0...",1,0.678587
